# Stock Analysis Dashboard

This notebook provides tools for analyzing stock data using various financial APIs and libraries.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import yfinance as yf
import yahoo_fin as yf_fin
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import cufflinks as cf
cf.go_offline()

print("Libraries imported successfully!")

In [ ]:
# Function to get stock data
def get_stock_data(ticker, period='1y'):
    """
    Fetch stock data for a given ticker and period
    
    Args:
        ticker (str): Stock ticker symbol
        period (str): Time period ('1d', '5d', '1mo', '3mo', '6mo', '1y', '2y', '5y', '10y', 'ytd', 'max')
    
    Returns:
        pandas.DataFrame: Stock data
    """
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period=period)
        return data
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return None

In [ ]:
# Function to plot stock price
def plot_stock_price(data, ticker):
    """
    Create an interactive plot for stock price
    """
    if data is None:
        print("No data to plot")
        return
    
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=(f'{ticker} Stock Price', 'Volume'),
        row_width=[0.2, 0.7]
    )
    
    # Candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data['Open'],
            high=data['High'],
            low=data['Low'],
            close=data['Close'],
            name='Price'
        ),
        row=1, col=1
    )
    
    # Volume chart
    fig.add_trace(
        go.Bar(
            x=data.index,
            y=data['Volume'],
            name='Volume',
            marker_color='rgba(0,0,255,0.3)'
        ),
        row=2, col=1
    )
    
    fig.update_layout(
        title=f'{ticker} Stock Analysis',
        yaxis_title='Price',
        xaxis_rangeslider_visible=False,
        height=800
    )
    
    fig.show()

In [ ]:
# Example usage
# Analyze a stock (change ticker as needed)
ticker_symbol = 'AAPL'  # Apple Inc.
stock_data = get_stock_data(ticker_symbol, period='1y')

if stock_data is not None:
    print(f"Data for {ticker_symbol}:")
    print(stock_data.tail())
    print(f"\nData shape: {stock_data.shape}")
    
    # Plot the data
    plot_stock_price(stock_data, ticker_symbol)

In [ ]:
# Multiple stocks comparison
def compare_stocks(tickers, period='1y'):
    """
    Compare multiple stocks
    """
    data = {}
    
    for ticker in tickers:
        stock_data = get_stock_data(ticker, period)
        if stock_data is not None:
            # Normalize to percentage change from first day
            data[ticker] = (stock_data['Close'] / stock_data['Close'].iloc[0] - 1) * 100
    
    if data:
        df_comparison = pd.DataFrame(data)
        
        fig = px.line(
            df_comparison, 
            title=f'Stock Performance Comparison ({period})',
            labels={'value': 'Return (%)', 'index': 'Date'}
        )
        fig.show()
        
        return df_comparison
    else:
        return None

In [ ]:
# Compare multiple stocks
stocks_to_compare = ['AAPL', 'GOOGL', 'MSFT', 'TSLA']
comparison_data = compare_stocks(stocks_to_compare, period='1y')

In [ ]:
# Get stock statistics
def get_stock_stats(ticker):
    """
    Get detailed stock statistics
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        key_stats = {
            'Company Name': info.get('longName', 'N/A'),
            'Sector': info.get('sector', 'N/A'),
            'Market Cap': info.get('marketCap', 'N/A'),
            'P/E Ratio': info.get('trailingPE', 'N/A'),
            'Dividend Yield': info.get('dividendYield', 'N/A'),
            '52 Week High': info.get('fiftyTwoWeekHigh', 'N/A'),
            '52 Week Low': info.get('fiftyTwoWeekLow', 'N/A'),
            'Beta': info.get('beta', 'N/A')
        }
        
        return pd.DataFrame(list(key_stats.items()), columns=['Metric', 'Value'])
    except Exception as e:
        print(f"Error getting stats for {ticker}: {e}")
        return None

In [ ]:
# Display statistics for a stock
stats_df = get_stock_stats('AAPL')
if stats_df is not None:
    display(stats_df)